# Keyloop Scenario A · Reviewer Demo

**Unified Service Scheduler** — Node.js / Hono / SQLite3, Vue 3 + Tailwind Play CDN. This notebook performs real scheduling demonstrations against a **fresh private sandbox** copied from the source checkout; it **never connects to the existing shared live database**.

- Source checkout must already be present at `/kaggle/working/Keyloop-Scenario-A-Unified-Service-Scheduler` (or set `KEYLOOP_NOTEBOOK_SOURCE` to an authorized copy).
- Setup creates a unique `/kaggle/working/keyloop-a-reviewer-isolated-*` directory excluding live SQLite, token files, secrets, and caches. It installs npm dependencies, seeds its own SQLite, starts its own localhost backend on a random non-3000 port, and runs all tests and HTTP demos there.
- Final cell checks sandbox SQLite integrity and stops only the notebook's server. If a cell fails, the notebook kernel's cleanup handler terminates that server.
- **No Docker, no Google Drive, no write token, and no public tunnel are required.** Sandbox artifacts can be inspected after completion. Avoid sharing any real personal credentials in notebook outputs.

## M0 / M1 · Isolated bootstrap and reproducible seed

Source checkout must exist in `/kaggle/working`. The setup cell copies source files to a new **disposable** folder, excludes live data and secrets, and uses a separate SQLite and server port. It deliberately never runs migration/seed in the original checkout.

In [ ]:
from pathlib import Path
import subprocess, urllib.request, urllib.error, json, os, time, uuid, datetime
import shutil, socket, signal, atexit

# The author/source checkout is READ ONLY to this notebook.
SOURCE_ROOT = Path(os.environ.get(
    "KEYLOOP_NOTEBOOK_SOURCE",
    "/kaggle/working/Keyloop-Scenario-A-Unified-Service-Scheduler"
)).resolve()
assert (SOURCE_ROOT / "package-lock.json").is_file(), f"Missing source: {SOURCE_ROOT}"

# Always make a NEW isolated directory, without copying live SQLite or secrets.
SANDBOX_PARENT = Path("/kaggle/working").resolve()
ROOT = SANDBOX_PARENT / ("keyloop-a-reviewer-isolated-" + uuid.uuid4().hex[:10])
assert ROOT != SOURCE_ROOT and not ROOT.is_relative_to(SOURCE_ROOT)
assert not ROOT.exists()
exclude = shutil.ignore_patterns(
    ".git", ".runtime", ".env", ".env.*", "node_modules",
    "data", "backups", "evidence", "__pycache__", "*.sqlite*",
    "*.log", "*.tar.gz", "*.zip", "*token*", "*secret*"
)
shutil.copytree(SOURCE_ROOT, ROOT, ignore=exclude)
(ROOT / "data").mkdir()
(ROOT / "evidence").mkdir()

# Choose a separate loopback port; never bind the shared public demo port 3000.
with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0))
    REVIEWER_PORT = probe.getsockname()[1]
assert REVIEWER_PORT != 3000
sandbox_env = os.environ.copy()
for forbidden in ("DEMO_WRITE_TOKEN", "PUBLIC_DEMO", "DATABASE_PATH", "HOST", "PORT"):
    sandbox_env.pop(forbidden, None)
sandbox_env.update(
    HOST="127.0.0.1",
    PORT=str(REVIEWER_PORT),
    DATABASE_PATH=str(ROOT / "data" / "scheduler.sqlite")
)
assert Path(sandbox_env["DATABASE_PATH"]).resolve().is_relative_to(ROOT)
server = None
def stop_reviewer_server():
    global server
    if server is not None and server.poll() is None:
        try:
            os.killpg(server.pid, signal.SIGTERM)
            server.wait(timeout=5)
        except Exception:
            try: server.kill()
            except Exception: pass
atexit.register(stop_reviewer_server)
def run(*argv):
    p = subprocess.run(argv, cwd=ROOT, env=sandbox_env,
                       text=True, capture_output=True)
    print("$", *argv, "status:", p.returncode, "\n", p.stdout[-2500:], p.stderr[-500:])
    assert p.returncode == 0, f"Command failed: {argv}"
print("REVIEWER_SANDBOX=", ROOT)
print("REVIEWER_LOOPBACK_PORT=", REVIEWER_PORT)
print("ISOLATED_DATABASE=", sandbox_env["DATABASE_PATH"])
run("npm", "ci")
run("npm", "run", "db:migrate")
run("npm", "run", "db:seed:rich")
run("npm", "run", "db:seed:demo")

## M2 / M3 · Automated verification

Tests cover resource qualification, full-duration intervals, no-overlap triggers, idempotency, HTTP 201/200/409, persistence and a **ready-barrier race across ten independent SQLite processes**.

In [ ]:
p=subprocess.run(["npm","test"],cwd=ROOT,env=sandbox_env,text=True,capture_output=True)
print(p.stdout[-7000:])
assert p.returncode==0, p.stderr[-1000:]

## Launch sandbox-only Hono server

This starts a new localhost server on the dynamically chosen `REVIEWER_PORT`, with `DATABASE_PATH` pointing to the isolated SQLite. It does not reuse port 3000 or attach to the shared public demo.

In [ ]:
BASE=f"http://127.0.0.1:{REVIEWER_PORT}"
def api(path,method="GET",payload=None,headers=None):
    req=urllib.request.Request(
        BASE+path,method=method,
        data=None if payload is None else json.dumps(payload).encode(),
        headers={"Content-Type":"application/json",**(headers or {})}
    )
    try:
        with urllib.request.urlopen(req,timeout=15) as r: return r.status,json.load(r)
    except urllib.error.HTTPError as e: return e.code,json.load(e)
log=open(ROOT/"evidence"/"notebook-server.log","a")
server=subprocess.Popen(
    ["node","src/server.js"],cwd=ROOT,env=sandbox_env,
    stdout=log,stderr=subprocess.STDOUT,start_new_session=True
)
log.close()
for _ in range(50):
    if server.poll() is not None:
        raise RuntimeError("Isolated server terminated at startup")
    try:
        if api("/readyz")[0]==200: break
    except Exception:
        pass
    time.sleep(0.2)
else:
    raise RuntimeError("Isolated backend readiness timeout")
assert api("/readyz")[0]==200
print("ISOLATED_BACKEND_READY=",BASE)

## Isolated HTTP proof · available → confirmed → idempotent replay → conflict → read

The following code inserts exactly one new **sandbox-only** synthetic booking and validates replay/conflict/read behavior. It does not touch live SQLite. Date selection is dynamic.

In [ ]:
from urllib.parse import urlencode
today=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).date()
date=(today+datetime.timedelta(days=(7-today.weekday())%7 or 7)).isoformat()
query=urlencode({"dealershipId":"dlr_001","serviceTypeId":"svc_brake","date":date})
s,found=api("/api/v1/availability?"+query)
assert s==200 and found["data"]["slots"]
body={"customerId":"cus_001","vehicleId":"veh_001","dealershipId":"dlr_001","serviceTypeId":"svc_brake","startAt":found["data"]["slots"][0]["startAt"]}
key="review-"+uuid.uuid4().hex
one,created=api("/api/v1/appointments","POST",body,{"Idempotency-Key":key})
two,replayed=api("/api/v1/appointments","POST",body,{"Idempotency-Key":key})
three,conflict=api("/api/v1/appointments","POST",body,{"Idempotency-Key":"review-"+uuid.uuid4().hex})
four,persisted=api("/api/v1/appointments/"+created["data"]["id"])
report={"status":[one,two,three,four],"appointmentId":created["data"]["id"],"sameId":created["data"]["id"]==replayed["data"]["id"]}
print(json.dumps(report,indent=2))
assert report["status"]==[201,200,409,200] and report["sameId"]

## Reviewer evidence and frontend entry point

The frontend is served on the exact same port, with browser imports to Vue ESM and Tailwind Play CDN. Browser keyboard, accessibility, responsiveness and styling should be reviewed independently.

In [ ]:
run("npm","run","evidence")
print("GET /:",urllib.request.urlopen(BASE+"/").status)
print("GET /js/main.js:",urllib.request.urlopen(BASE+"/js/main.js").status)
print("Review URL on this machine:",BASE)

## Video narrative (5–10 minutes)

1. Explain the scenario and three Keyloop acceptance criteria.
2. Architecture and technology decisions; note Tailwind Play CDN is demo-only and SQLite is one-host/one-writer.
3. Availability check requires both a qualified technician and a compatible bay across the full half-open interval.
4. Demonstrate actual booking, replay, conflict and persistence responses.
5. Show 10-process race, SQLite triggers, synthetic data and evidence.
6. Discuss AI collaboration: design steering, verification, Sunday fixture correction and UTC-offset contract fix.
7. Wrap up with limitations and lessons learned.

## Latest reviewer UX and simulation acceptance checks

The main page offers filtered appointment detail cards, a human-readable booking receipt, a two-request conflict test and a 10-independent-process race. Both simulators use isolated SQLite and do not change live appointments. The notebook booking HTTP proof above **does** create a synthetic live appointment; run intentionally and preserve backups.


In [ ]:
p=subprocess.run(["npm","run","evidence"],cwd=ROOT,env=sandbox_env,text=True,capture_output=True)
print(p.stdout[-4000:])
assert p.returncode==0, p.stderr[-1000:]
for idx,mode in enumerate(("conflict","concurrent")):
    if idx: time.sleep(21)  # backend 20-second simulation cooldown
    status,result=api("/api/v1/demo/simulate",method="POST",payload={"mode":mode})
    assert status==200, (status,result)
    data=result.get("data",result)
    print(mode,data.get("confirmed"),data.get("rejected"),data.get("databaseConfirmed"),data.get("integrity"))
    assert data["confirmed"]==1 and data["rejected"]==(1 if mode=="conflict" else 9)
    assert data["databaseConfirmed"]==1 and data["integrity"]=="ok"

## Manual reviewer QA (human verification required)

Automated browser and accessibility results are separate from screen-reader and real-device acceptance. No physical Android or manual Narrator verification is implied by executing this notebook. Consult `docs/UI13-REVIEWER-READINESS-20261010.md` in the source checkout.

## T21 operational acceptance evidence

These acceptance tests run on disposable SQLite files only. They cover 16 KiB JSON body size enforcement, SQLite write-lock contention returning HTTP 503, correlation of request IDs and a demo-token canary redaction check. Full route metrics and manual accessibility checks are separately pending.


In [ ]:
p=subprocess.run(["node","--test","tests/body-limit-and-contention.test.js","tests/observability-security.test.js"],cwd=ROOT,env=sandbox_env,text=True,capture_output=True)
print(p.stdout[-5000:])
assert p.returncode==0,p.stderr[-2000:]
assert "# fail 0" in p.stdout or "ℹ fail 0" in p.stdout

## Isolated run final verification and teardown

This checks that the review appointment was written ONLY to the sandbox, verifies SQLite integrity, and shuts down the isolated server. The live demo database was never passed to the notebook.


In [ ]:
import sqlite3
with sqlite3.connect(str(ROOT/"data"/"scheduler.sqlite")) as db:
    integrity=db.execute("PRAGMA integrity_check").fetchone()[0]
    appointments=db.execute("SELECT COUNT(*) FROM appointments").fetchone()[0]
    no_overlap=not db.execute("""
       SELECT 1 FROM appointments a JOIN appointments b
         ON a.id < b.id AND a.status='confirmed' AND b.status='confirmed'
         AND a.start_at_ms < b.end_at_ms AND b.start_at_ms < a.end_at_ms
         AND (a.technician_id=b.technician_id OR a.service_bay_id=b.service_bay_id)
       LIMIT 1
    """).fetchone()
assert integrity=="ok" and no_overlap and appointments>0
stop_reviewer_server()
assert server.poll() is not None
print("NOTEBOOK_ISOLATED_DB=PASS", "integrity="+integrity,
      "appointments="+str(appointments), "no_overlap="+str(no_overlap))
print("ISOLATED_SERVER_STOPPED=PASS")
